# HealthSynth Getting Started

This notebook generates a synthetic healthcare commercial analytics dataset and explores it using pandas and DuckDB.

We will look at:

- generated tables
- launch adoption curve
- prescriptions by product
- prescriptions by HCP decile
- specialty-product affinity


In [ ]:
from healthsynth.generator import generate

datasets = generate(
    hcps=1000,
    years=3,
    output_dir="../output",
    seed=42,
)

datasets.keys()


In [ ]:
hcp_master = datasets["hcp_master"]
product = datasets["product"]
call_activity = datasets["call_activity"]
prescriptions = datasets["prescriptions"]

print(hcp_master.shape)
print(product.shape)
print(call_activity.shape)
print(prescriptions.shape)


In [ ]:
product

In [ ]:
import pandas as pd

rx = prescriptions.copy()
rx["rx_date"] = pd.to_datetime(rx["rx_date"])

monthly_nrx = (
    rx.groupby("rx_date")["nrx"]
    .sum()
    .reset_index()
)

monthly_nrx.head()


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(monthly_nrx["rx_date"], monthly_nrx["nrx"])
plt.title("New Product Launch Adoption Curve")
plt.xlabel("Month")
plt.ylabel("Total NRx")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
rx_by_product = (
    prescriptions.merge(product[["product_id", "product_name"]], on="product_id")
    .groupby("product_name")["nrx"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

rx_by_product


In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(rx_by_product["product_name"], rx_by_product["nrx"])
plt.title("Total NRx by Product")
plt.xlabel("Product")
plt.ylabel("Total NRx")
plt.tight_layout()
plt.show()


In [ ]:
rx_with_hcp = prescriptions.merge(
    hcp_master[["hcp_id", "decile", "segment", "specialty"]],
    on="hcp_id",
)

rx_by_decile = (
    rx_with_hcp.groupby("decile")["nrx"]
    .sum()
    .reset_index()
)

rx_by_decile


In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(rx_by_decile["decile"], rx_by_decile["nrx"])
plt.title("Total NRx by HCP Decile")
plt.xlabel("HCP Decile")
plt.ylabel("Total NRx")
plt.tight_layout()
plt.show()


In [ ]:
specialty_product = (
    rx_with_hcp
    .merge(product[["product_id", "product_name"]], on="product_id")
    .groupby(["specialty", "product_name"])["nrx"]
    .sum()
    .reset_index()
    .sort_values(["specialty", "nrx"], ascending=[True, False])
)

specialty_product.head(20)


In [ ]:
import duckdb

con = duckdb.connect("../output/healthsynth.duckdb")

con.execute("SHOW TABLES").fetchdf()


In [ ]:
con.execute("""
SELECT
    p.product_name,
    SUM(rx.nrx) AS total_nrx,
    SUM(rx.trx) AS total_trx
FROM prescriptions rx
JOIN product p
    ON rx.product_id = p.product_id
GROUP BY p.product_name
ORDER BY total_nrx DESC
""").fetchdf()


## What this shows

HealthSynth is not only creating tables.

It is creating a small commercial analytics environment where:

- HCP segments affect call activity
- Calls influence prescriptions with lagged response
- Prescription response has diminishing returns
- Seasonality affects activity
- Products align with specialties
- DuckDB output is immediately queryable
